# 03 - Full Vehicle + ANPR Pipeline

End-to-end over a video: vehicle detect -> track -> plate recognize -> temporal vote -> speed via two virtual lines -> events in the backend JSON contract -> annotated video + JSONL/CSV.


## Setup


In [ ]:
import os, sys, pathlib

# --- point these at the real locations on Colab -----------------------------
AI_DIR = os.environ.get("TRAFFIQ_AI_DIR", "/content/traffIQ/ai")
WEIGHTS_DIR = os.environ.get("TRAFFIQ_WEIGHTS_DIR", "")

# /content/implementation is where notebook 00 wrote the modules (or your
# cloned/Mounted repo if you prefer to import from there instead).
sys.path.insert(0, "/content/implementation")
os.environ["TRAFFIQ_AI_DIR"] = str(AI_DIR)

from pipeline.config import Config, colab_config

if WEIGHTS_DIR:
    cfg = colab_config(weights_dir=WEIGHTS_DIR, ai_dir=AI_DIR)
else:
    cfg = Config(ai_dir=AI_DIR)
cfg.validate(require_all=False)
print("ai_dir      :", cfg.ai_dir)
print("ref_repo    :", cfg.ref_repo_dir)
print("plate_weights:", cfg.plate_weights)
print("vehicle_weights:", cfg.vehicle_weights)


## 1. Choose the demo clip + camera calibration


In [ ]:
import glob
VIDEO = glob.glob("/content/[Hh]ighway*.mp4")
VIDEO += glob.glob("/content/drive/MyDrive/**/[Hh]ighway*.mp4", recursive=True)
assert VIDEO, "drop a highway mp4 into /content first"
video_path = VIDEO[0]
print("video:", video_path)

# Calibration per camera (edit for your clip):
cfg.line_a_y = 198
cfg.line_b_y = 268
cfg.line_offset = 6
cfg.distance_meters = 10.0
cfg.direction_ab = "NORTH"   # crossing A->B
cfg.direction_ba = "SOUTH"   # crossing B->A
cfg.camera_id = "CAM-007"
cfg.speed_estimated = True
print("calibration set")


## 2. Build the pipeline


In [ ]:
from pipeline.pipeline import DetectionPipeline, VideoProcessor


## 3. Run the video


In [ ]:
import os, pathlib
out_dir = pathlib.Path("/content/outputs")
out_dir.mkdir(exist_ok=True)
cfg.output_dir = out_dir
vp = VideoProcessor(cfg)
events = vp.run(
    video_path,
    output_video=str(out_dir / "annotated.mp4"),
    jsonl_out=str(out_dir / "events.jsonl"),
    csv_out=str(out_dir / "events.csv"),
    slice_frames=None,   # e.g. 400 to test quickly
)
print("events:", len(events))


## 4. Inspect the emitted events


In [ ]:
import json
for e in events[:5]:
    print(json.dumps(e, indent=2))
print("...\nTOTAL EVENTS:", len(events))


## 5. Contract + summary metrics


In [ ]:
import pandas as pd
from collections import Counter
df = pd.read_csv(str(out_dir / "events.csv")) if events else pd.DataFrame()
if not df.empty:
    print("directions:", Counter(df.speed_direction).most_common())
    print("speed km/h  : min/max/mean =", round(df.speed_kmh.min(),1),
          round(df.speed_kmh.max(),1), round(df.speed_kmh.mean(),1))
    print("plate formats valid:", int(df.plate_format_valid.sum()), "/", len(df))
    print(df[["event_id", "local_track_id", "vehicle_type", "plate_text", "speed_kmh", "speed_direction"]].head(10))
else:
    print("No events - see docs/TROUBLESHOOTING.md section 4.")


## 6. Annotated video preview


In [ ]:
from IPython.display import Video
if pathlib.Path(str(out_dir / "annotated.mp4")).exists():
    Video(str(out_dir / "annotated.mp4"), width=720)
